# FLEO-FER — clean Kaggle run (persist-safe, quota-friendly)

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **ON**.
**Add Input:** attach `msambare/fer2013` **and** `shuvoalok/raf-db-dataset`.

**Don't lose your work:** use **Save Version -> Save & Run All (Commit)** — it runs top-to-bottom and *saves all output permanently*. 60 epochs fit inside the 12h limit.

**Never-lose-everything design:**
- FER2013 outputs are zipped as soon as they exist.
- The RAF-DB step is **time-guarded**: it only starts if enough budget remains before the 12h kernel wall, otherwise it is skipped and the commit still finishes cleanly with all FER2013 outputs persisted.
- Result cells never raise on missing files, so a partial run can never fail the whole commit.

FLEO trains with dropout + stronger orthogonality + cosine LR (anti-overfit) automatically.

## 1. Clone + install

In [ ]:
import time, pathlib
pathlib.Path('/kaggle/working/t0.txt').write_text(str(time.time()))
%cd /kaggle/working
!rm -rf FLEO
!git clone https://github.com/olfa-askri/FLEO.git
%cd /kaggle/working/FLEO
!pip install -q ultralytics onnx onnxruntime onnxscript
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Auto-find the datasets + prepare (works whatever the mount name)

In [ ]:
import glob, os
def find_root(keys):
    for p in sorted(glob.glob('/kaggle/input/*'))+sorted(glob.glob('/kaggle/input/*/*'))+sorted(glob.glob('/kaggle/input/*/*/*')):
        if os.path.isdir(p) and any(k in p.lower() for k in keys):
            return p
    return None
fer = find_root(['fer2013','fer-2013'])
raf = find_root(['raf-db','rafdb','raf_db'])
print('FER root:', fer); print('RAF root:', raf)
assert fer and raf, 'Attach msambare/fer2013 and shuvoalok/raf-db-dataset via Add Input!'

In [ ]:
!python -m data.prepare_fer2013 --src "{fer}" --out /kaggle/working/FLEO/datasets/fer2013
!python -m data.prepare_rafdb   --src "{raf}" --out /kaggle/working/FLEO/datasets/rafdb
!ls -l /kaggle/working/FLEO/datasets/fer2013/data.yaml /kaggle/working/FLEO/datasets/rafdb/data.yaml

## 3. FER2013 — baseline + FLEO + export R1/R2/R3 + Delta_fold  (60 epochs)

In [ ]:
!python -m scripts.run_matrix --data /kaggle/working/FLEO/datasets/fer2013/data.yaml --dataset fer2013 --seeds 0 --epochs 60 --imgsz 160 --batch 64 --device 0 --project /kaggle/working/FLEO/runs/fer2013

## 4. FER2013 — Delta_fold, FLEO macro-F1, and baseline accuracy

In [ ]:
import json, os
def show_json(path, title):
    print('=== %s ===' % title)
    print(json.dumps(json.load(open(path)), indent=2) if os.path.exists(path) else 'MISSING: ' + path)
show_json('/kaggle/working/FLEO/results/deltas_fer2013.json', 'FER2013 Delta_fold + accuracy (full/folded/householder)')
Wf = '/kaggle/working/FLEO/runs/fer2013/fleo_seed0/weights/best.pt'
Wb = '/kaggle/working/FLEO/runs/fer2013/baseline_seed0/weights/best.pt'
print('\n=== FLEO macro-F1 ===')
!python -m scripts.deltas --data /kaggle/working/FLEO/datasets/fer2013/data.yaml --dataset fer2013 --imgsz 160 --device 0 --weights {Wf} --metric macro_f1 --out /kaggle/working/FLEO/results/macro_f1_fer2013.json
print('\n=== BASELINE (accuracy + macro-F1) ===')
!python -m scripts.evaluate --data /kaggle/working/FLEO/datasets/fer2013/data.yaml --weights {Wb} --imgsz 160 --device 0 --out /kaggle/working/FLEO/results/baseline_fer2013.json

## 5. Zip FER2013 — persisted no matter what happens later

In [ ]:
%cd /kaggle/working/FLEO
!zip -qr /kaggle/working/fleo_fer2013.zip runs export results
!ls -lh /kaggle/working/fleo_fer2013.zip

## 6. RAF-DB — full matrix (60 epochs, time-guarded)

In [ ]:
import time, pathlib
t0 = float(pathlib.Path('/kaggle/working/t0.txt').read_text())
left_h = 11.5 - (time.time() - t0) / 3600
print('Budget left before the 12h wall (0.5h safety margin): %.2f h' % left_h)
if left_h > 3.5:
    !python -m scripts.run_matrix --data /kaggle/working/FLEO/datasets/rafdb/data.yaml --dataset rafdb --seeds 0 --epochs 60 --imgsz 160 --batch 64 --device 0 --project /kaggle/working/FLEO/runs/rafdb
else:
    print('SKIPPED RAF-DB: not enough time left. FER2013 outputs are zipped and safe; rerun RAF-DB in a fresh session.')

## 7. RAF-DB — the numbers (skipped gracefully if the guard fired)

In [ ]:
import json, os
p = '/kaggle/working/FLEO/results/deltas_rafdb.json'
if os.path.exists(p):
    print('=== RAF-DB Delta_fold + accuracy ===')
    print(json.dumps(json.load(open(p)), indent=2))
    Wb = '/kaggle/working/FLEO/runs/rafdb/baseline_seed0/weights/best.pt'
    print('\n=== BASELINE ===')
    !python -m scripts.evaluate --data /kaggle/working/FLEO/datasets/rafdb/data.yaml --weights {Wb} --imgsz 160 --device 0 --out /kaggle/working/FLEO/results/baseline_rafdb.json
else:
    print('RAF-DB was skipped by the time guard - no numbers this run.')

## 8. Final bundle

In [ ]:
%cd /kaggle/working/FLEO
!zip -qr /kaggle/working/fleo_all.zip runs export results
!ls -lh /kaggle/working/*.zip
print('DONE. Download the zips from the Output panel once the commit finishes.')